# OFL competency question evaluation

This notebook evaluates the Comprehensive Ontology of Flaps in Reconstructive Surgery (OFL) 2.2.1 against its competency questions (CQ1–CQ17) and five task-based use cases, with SPARQL queries over a synthetic ABox. The counts it reports are those of Supplementary Table S2.

| File | Role |
|------|------|
| `ontologies/ofl_2.2.1.owl` | OFL 2.2.1 |
| `ontologies/imports/` | Modules imported by OFL (FMA OBO, GO, OBI, RO) and COB 2025-12-15 |
| `ontologies/bfo-core.owl` | BFO 2020, imported whole; downloaded from its PURL on the first run |
| `ontologies/ofl_2.2.1_abox_inferred.owl` (or `.owl.gz`) | Synthetic ABox (86,030 individuals), materialised with OWL RL; attached to the v2.2.1 release |

**Reproduce:** install `pyoxigraph` and `pandas`, put the ABox file in `ontologies/`, then run all cells from top to bottom.

For every query the notebook reports the number of result rows and the number of distinct individuals returned (flaps; anastomoses for CQ16; components for Task 5), and shows the first ten rows.

In [1]:
import gzip
import re
import urllib.request
from pathlib import Path

import pandas as pd
import pyoxigraph as ox
from IPython.display import display


class _Namespace(str):
    def __getattr__(self, name):
        return self + name


OFL = _Namespace('https://purl.bioontology.org/ontology/OFL/')
OBO = _Namespace('http://purl.obolibrary.org/obo/')

PREFIXES = """
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:     <http://www.w3.org/2002/07/owl#>
PREFIX obo:     <http://purl.obolibrary.org/obo/>
PREFIX ofl:     <https://purl.bioontology.org/ontology/OFL/>
PREFIX dcterms: <http://purl.org/dc/terms/>
"""

ONT     = Path('..') / 'ontologies'
TBOX    = ONT / 'ofl_2.2.1.owl'
MODULES = [ONT / 'imports' / f for f in ('fma_obo_import.owl', 'go_import.owl', 'obi_import.owl', 'ro_import.owl', 'cob.owl')]
UPPER   = {   # imported whole; downloaded once from its PURL
    ONT / 'bfo-core.owl': 'http://purl.obolibrary.org/obo/bfo/2020/bfo-core.owl',
}
ABOX    = ONT / 'ofl_2.2.1_abox_inferred.owl'
if not ABOX.exists():
    ABOX = ONT / 'ofl_2.2.1_abox_inferred.owl.gz'

for path, url in UPPER.items():
    if not path.exists():
        urllib.request.urlretrieve(url, path)

store = ox.Store()
for path in [TBOX, *MODULES, *UPPER, ABOX]:
    with (gzip.open if path.suffix == '.gz' else open)(path, 'rb') as f:
        store.bulk_load(f, format=ox.RdfFormat.RDF_XML)
print(f'Total: {len(store):,} triples')

Total: 1,413,365 triples


In [2]:
RESULTS = {}


def _value(term):
    return term.value if term is not None else "—"


def run(sparql, title="", key=None, count_var="?flap", show=10):
    """Run a query, report rows and distinct individuals, and show the first rows.

    The counted variable is added to the projection if the query does not select it;
    for a SELECT without DISTINCT this does not change the number of rows.
    """
    m = re.search(r'SELECT\s+(DISTINCT\s+)?(.*?)\s+WHERE', sparql, re.S)
    distinct, cols = m.group(1) or '', m.group(2).split()
    if count_var not in cols:
        assert not distinct, 'add the counted variable to a SELECT DISTINCT'
        sparql = sparql.replace(m.group(0), f'SELECT {count_var} {m.group(2)}\nWHERE', 1)
    rows = list(store.query(PREFIXES + sparql))
    n = len({r[count_var[1:]] for r in rows})
    if key:
        RESULTS[key] = {'Individuals returned': n, 'Rows': len(rows)}
    names = [c[1:] for c in cols]
    df = pd.DataFrame([{c: _value(r[c]) for c in names} for r in rows[:show]], columns=names)
    print(title)
    print(f"  {n:,} individuals returned, {len(rows):,} rows" + (f" (first {show} shown)" if len(rows) > show else ""))
    display(df)

In [3]:
SUBCLASS_OF = 'http://www.w3.org/2000/01/rdf-schema#subClassOf'


def _select(query, var):
    return [s[var].value for s in store.query(query)]


# Only classes that occur as types in the ABox are put into VALUES
_all_types = set(_select('SELECT DISTINCT ?t WHERE { ?x a ?t FILTER(isIRI(?t)) }', 't'))


def _subclasses(root):
    """root and all its transitive subclasses."""
    return {root} | set(_select(f'SELECT DISTINCT ?c WHERE {{ ?c <{SUBCLASS_OF}>* <{root}> FILTER(isIRI(?c)) }}', 'c'))


def _cls_vals(*roots):
    """Transitive subclasses of roots that appear as types in the ABox."""
    subs = set(roots)
    for r in roots:
        subs |= _subclasses(r)
    used = subs & _all_types or subs
    return " ".join(f"<{c}>" for c in sorted(used))


def _pair_vals(*roots):
    """(root, subclass) pairs where the subclass appears as a type in the ABox."""
    pairs, seen = [], set()
    for r in roots:
        for sub in sorted(_subclasses(r) & _all_types):
            if (r, sub) not in seen:
                seen.add((r, sub))
                pairs.append(f"(<{r}> <{sub}>)")
    return " ".join(pairs) or " ".join(f"(<{r}> <{r}>)" for r in roots)

flap_cls       = _cls_vals(OFL.OFLID10002)
pedicle_cls    = _cls_vals(OFL.OFLID106008)
mn_cls         = _cls_vals(OFL.OFLID10135)
distance_cls   = _cls_vals(OFL.OFLID10086)
chimeric_cls   = _cls_vals(OFL.OFLID1000093)
island_cls     = _cls_vals(OFL.OFLID10115)
preharvest_cls      = _cls_vals(OFL.OFLID10045)
insertion_prep_cls  = _cls_vals(OFL.OFLID130001)
skingraft_cls       = _cls_vals(OFL.OFLID120066)
anast_cls      = _cls_vals(OFL.OFLID1000170, OFL.OFLID1000175)
pedicled_cls   = _cls_vals(OFL.OFLID1000138)
fma_vessel_cls = _cls_vals(OBO.FMA_86187, OBO.FMA_86188)

nakajima_pairs = _pair_vals(OFL.OFLID10075, OFL.OFLID10076, OFL.OFLID10078)
transfer_pairs = _pair_vals(OFL.OFLID1000137, OFL.OFLID1000138)
vasc_pairs     = _pair_vals(OFL.OFLID10004, OFL.OFLID10077, OFL.OFLID10076,
                             OFL.OFLID10075, OFL.OFLID10078)
flow_pairs     = _pair_vals(OFL.OFLID10113, OFL.OFLID10123)
movement_pairs = _pair_vals(OFL.OFLID10014, OFL.OFLID10067, OFL.OFLID10070, OFL.OFLID10132)

_mn_direct     = _select(f'SELECT DISTINCT ?c WHERE {{ ?c <{SUBCLASS_OF}> <{OFL.OFLID10135}> FILTER(isIRI(?c)) }}', 'c')
mn_task3_pairs = _pair_vals(*_mn_direct)

# backward-compat alias used by CQ1
flap_cls_values = flap_cls

print("Class sets precomputed.")

Class sets precomputed.


---
## Competency Questions

### CQ1 — Anatomical Composition
*What anatomical structures compose the flap?*

In [4]:
_cq1 = """
SELECT ?flaplabel ?componentLabel ?typeLabel
WHERE {
  VALUES ?cls { FLAP_CLS_PLACEHOLDER }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?cls ;
        ofl:OFLID13296 ?component .
  OPTIONAL { ?flap      rdfs:label ?flaplabel }
  OPTIONAL { ?component rdfs:label ?componentLabel }
  OPTIONAL {
    ?component rdf:type ?t .
    FILTER(STRSTARTS(STR(?t), "http://purl.obolibrary.org/obo/FMA_"))
    OPTIONAL { ?t rdfs:label ?typeLabel }
  }
}
"""
run(_cq1.replace("FLAP_CLS_PLACEHOLDER", flap_cls_values),
    key="CQ1", title="CQ1 ABox — anatomical components of generated flaps")

CQ1 ABox — anatomical components of generated flaps
  3,948 individuals returned, 231,692 rows (first 10 shown)


,flaplabel,componentLabel,typeLabel
0,Right Vertical rectus abdominis flap of patien...,Flap tissue of Right Vertical rectus abdominis...,Anatomical structure
1,Left Vertical rectus abdominis flap of patient...,Flap tissue of Left Vertical rectus abdominis ...,Anatomical structure
2,Right Oblique rectus abdominis flap of patient...,Flap tissue of Right Oblique rectus abdominis ...,Anatomical structure
3,Right Oblique rectus abdominis flap of patient...,Flap tissue of Right Oblique rectus abdominis ...,Anatomical structure
4,Left Vertical rectus abdominis flap of patient...,Flap tissue of Left Vertical rectus abdominis ...,Anatomical structure
5,Left Oblique rectus abdominis flap of patient ...,Flap tissue of Left Oblique rectus abdominis f...,Anatomical structure
6,Right Vertical rectus abdominis flap of patien...,Flap tissue of Right Vertical rectus abdominis...,Anatomical structure
7,Left Oblique rectus abdominis flap of patient ...,Flap tissue of Left Oblique rectus abdominis f...,Anatomical structure
8,Left Oblique rectus abdominis flap of patient ...,Flap tissue of Left Oblique rectus abdominis f...,Anatomical structure
9,Left Vertical rectus abdominis flap of patient...,Flap tissue of Left Vertical rectus abdominis ...,Anatomical structure


### CQ2 — Size
*What is the size of the flap?*

In [5]:
_cq2 = """
SELECT ?flapLabel ?qualityLabel
WHERE {
  VALUES ?cls { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?cls ;
        obo:RO_0000086 ?quality .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?quality rdfs:label ?qualityLabel }
}
"""
run(_cq2.replace("CLS", flap_cls),
    key="CQ2", title="CQ2 ABox — flap individuals with size qualities (volume / mass)")

CQ2 ABox — flap individuals with size qualities (volume / mass)
  3,948 individuals returned, 191,356 rows (first 10 shown)


,flapLabel,qualityLabel
0,Right Vertical rectus abdominis flap of patien...,Volume of Right Vertical rectus abdominis flap...
1,Right Vertical rectus abdominis flap of patien...,Mass of Right Vertical rectus abdominis flap o...
2,Left Vertical rectus abdominis flap of patient...,Volume of Left Vertical rectus abdominis flap ...
3,Left Vertical rectus abdominis flap of patient...,Mass of Left Vertical rectus abdominis flap of...
4,Right Oblique rectus abdominis flap of patient...,Volume of Right Oblique rectus abdominis flap ...
5,Right Oblique rectus abdominis flap of patient...,Mass of Right Oblique rectus abdominis flap of...
6,Right Oblique rectus abdominis flap of patient...,Volume of Right Oblique rectus abdominis flap ...
7,Right Oblique rectus abdominis flap of patient...,Mass of Right Oblique rectus abdominis flap of...
8,Left Vertical rectus abdominis flap of patient...,Volume of Left Vertical rectus abdominis flap ...
9,Left Vertical rectus abdominis flap of patient...,Mass of Left Vertical rectus abdominis flap of...


### CQ3 — Vessel Connection
*To which vessels is the flap connected?*

In [6]:
_cq3 = """
SELECT ?flapLabel ?pedicleLabel ?vesselLabel
WHERE {
  VALUES ?cls { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?cls ;
        ofl:OFLID13296 ?pedicle .
  ?pedicle ofl:OFLID13296 ?vessel .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?pedicle rdfs:label ?pedicleLabel }
  OPTIONAL { ?vessel  rdfs:label ?vesselLabel }
}
"""
run(_cq3.replace("CLS", _cls_vals(OFL.OFLID10002)),
    key="CQ3", title="CQ3 ABox — flap individuals with their pedicle part and vessels")

CQ3 ABox — flap individuals with their pedicle part and vessels
  302 individuals returned, 27,614 rows (first 10 shown)


,flapLabel,pedicleLabel,vesselLabel
0,Left Wrap around flap of patient 1133,Flap pedicle of Left Wrap around flap of patie...,Left vessel 44440 of patient 1133
1,Left Wrap around flap of patient 1133,Flap pedicle of Left Wrap around flap of patie...,Left vessel 44647 of patient 1133
2,Left Wrap around flap of patient 1133,Flap pedicle of Left Wrap around flap of patie...,Pedicle artery of Left Wrap around flap of pat...
3,Left Wrap around flap of patient 1133,Flap pedicle of Left Wrap around flap of patie...,Pedicle vein of Left Wrap around flap of patie...
4,Right Wrap around flap of patient 1137,Flap pedicle of Right Wrap around flap of pati...,Right vessel 44440 of patient 1137
5,Right Wrap around flap of patient 1137,Flap pedicle of Right Wrap around flap of pati...,Right vessel 44647 of patient 1137
6,Right Wrap around flap of patient 1137,Flap pedicle of Right Wrap around flap of pati...,Pedicle artery of Right Wrap around flap of pa...
7,Right Wrap around flap of patient 1137,Flap pedicle of Right Wrap around flap of pati...,Pedicle vein of Right Wrap around flap of pati...
8,Left Wrap around flap of patient 1134,Flap pedicle of Left Wrap around flap of patie...,Left vessel 44440 of patient 1134
9,Left Wrap around flap of patient 1134,Flap pedicle of Left Wrap around flap of patie...,Left vessel 44647 of patient 1134


### CQ4 — Mathes and Nahai Classification
*What is the Mathes and Nahai classification of the muscle flap?*

In [7]:
_cq4a = """
SELECT ?flapLabel ?originLabel
WHERE {
  VALUES ?cls { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?cls ;
        ofl:OFLID12003 ?origin .
  OPTIONAL { ?flap   rdfs:label ?flapLabel }
  OPTIONAL { ?origin rdfs:label ?originLabel }
}
"""
run(_cq4a.replace("CLS", _cls_vals(OFL.OFLID10135)),
    key="CQ4", title="CQ4 ABox — Mathes-Nahai classified flap individuals with their flap origin")

CQ4 ABox — Mathes-Nahai classified flap individuals with their flap origin
  1,240 individuals returned, 2,888 rows (first 10 shown)


,flapLabel,originLabel
0,Left Rectus abdominis flap with fascia of rect...,Donor site of Left Rectus abdominis flap with ...
1,Right Rectus abdominis flap with fascia of rec...,Donor site of Right Rectus abdominis flap with...
2,Left Rectus abdominis flap with fascia of rect...,Donor site of Left Rectus abdominis flap with ...
3,Left Rectus abdominis flap with fascia of rect...,Donor site of Left Rectus abdominis flap with ...
4,Left Rectus abdominis flap with fascia of rect...,Donor site of Left Rectus abdominis flap with ...
5,Right Rectus abdominis flap with fascia of rec...,Donor site of Right Rectus abdominis flap with...
6,Left Rectus abdominis flap with fascia of rect...,Donor site of Left Rectus abdominis flap with ...
7,Left Rectus abdominis flap with fascia of rect...,Donor site of Left Rectus abdominis flap with ...
8,Left Rectus abdominis flap with fascia of rect...,Donor site of Left Rectus abdominis flap with ...
9,Right Rectus abdominis flap with fascia of rec...,Donor site of Right Rectus abdominis flap with...


In [8]:
_cq4b = """
SELECT ?flapLabel ?typeLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?typeLabel }
}
"""
run(_cq4b.replace("CLS", mn_cls),
    title="CQ4 ABox — flap individuals by Mathes-Nahai type")

CQ4 ABox — flap individuals by Mathes-Nahai type
  1,246 individuals returned, 2,856 rows (first 10 shown)


,flapLabel,typeLabel
0,Left Rectus abdominis flap with fascia of rect...,Rectus abdominis flap with fascia of rectus ab...
1,Right Rectus abdominis flap with fascia of rec...,Rectus abdominis flap with fascia of rectus ab...
2,Left Rectus abdominis flap with fascia of rect...,Rectus abdominis flap with fascia of rectus ab...
3,Left Rectus abdominis flap with fascia of rect...,Rectus abdominis flap with fascia of rectus ab...
4,Left Rectus abdominis flap with fascia of rect...,Rectus abdominis flap with fascia of rectus ab...
5,Right Rectus abdominis flap with fascia of rec...,Rectus abdominis flap with fascia of rectus ab...
6,Left Rectus abdominis flap with fascia of rect...,Rectus abdominis flap with fascia of rectus ab...
7,Left Rectus abdominis flap with fascia of rect...,Rectus abdominis flap with fascia of rectus ab...
8,Left Rectus abdominis flap with fascia of rect...,Rectus abdominis flap with fascia of rectus ab...
9,Right Rectus abdominis flap with fascia of rec...,Rectus abdominis flap with fascia of rectus ab...


### CQ5 — Nakajima Classification
*What is the Nakajima classification of the flap?*

In [9]:
_cq5 = """
SELECT ?flapLabel ?nakajimaLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?nakajimaLabel }
}
"""
run(_cq5.replace("CLS", _cls_vals(OFL.OFLID10078)),
    key="CQ5", title="CQ5 ABox — perforator-based flap individuals (Nakajima)")

CQ5 ABox — perforator-based flap individuals (Nakajima)
  60 individuals returned, 180 rows (first 10 shown)


,flapLabel,nakajimaLabel
0,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
1,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
2,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
3,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
4,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...
5,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
6,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
7,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
8,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...
9,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...


### CQ6 — Transfer Distance
*Is the flap local, regional or distant?*

In [10]:
_cq6 = """
SELECT ?flapLabel ?distLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?distLabel }
}
"""
run(_cq6.replace("CLS", _cls_vals(OFL.OFLID10134)),
    key="CQ6", title="CQ6 ABox — local flap individuals (OFLID10134)")

CQ6 ABox — local flap individuals (OFLID10134)
  6 individuals returned, 6 rows


,flapLabel,distLabel
0,Right Local flaps of patient 1999,Local flaps
1,Right Local flaps of patient 2000,Local flaps
2,Right Local flaps of patient 1998,Local flaps
3,Left Local flaps of patient 1995,Local flaps
4,Left Local flaps of patient 1996,Local flaps
5,Left Local flaps of patient 1997,Local flaps


### CQ7 — Transfer Method
*Is the flap pedicled or free?*

In [11]:
_cq7 = """
SELECT ?flapLabel ?typeLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?typeLabel }
}
"""
run(_cq7.replace("CLS", _cls_vals(OFL.OFLID10191)),
    key="CQ7", title="CQ7 ABox — flap individuals by route of transfer (OFLID10191)")

CQ7 ABox — flap individuals by route of transfer (OFLID10191)
  140 individuals returned, 560 rows (first 10 shown)


,flapLabel,typeLabel
0,Right Pectoralis major flap with skin of anter...,Pectoralis major flap with skin of anterior chest
1,Left Pectoralis major flap with skin of anteri...,Pectoralis major flap with skin of anterior chest
2,Right Pectoralis major flap with skin of anter...,Pectoralis major flap with skin of anterior chest
3,Left Pectoralis major flap with skin of anteri...,Pectoralis major flap with skin of anterior chest
4,Right Pectoralis major flap with skin of anter...,Pectoralis major flap with skin of anterior chest
5,Right Pectoralis major flap with skin of anter...,Pectoralis major flap with skin of anterior chest
6,Right Pectoralis major flap with skin of anter...,Pectoralis major flap with skin of anterior chest
7,Left Pectoralis major flap with skin of anteri...,Pectoralis major flap with skin of anterior chest
8,Left Pectoralis major flap with skin of anteri...,Pectoralis major flap with skin of anterior chest
9,Left Pectoralis major flap with skin of anteri...,Pectoralis major flap with skin of anterior chest


### CQ8 — Vascular Pattern
*Is it a random pattern or an axial or perforator flap?*

In [12]:
_cq8 = """
SELECT ?flapLabel ?bloodSupplyLabel
WHERE {
  VALUES (?bloodSupplyCls ?t) { PAIRS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap           rdfs:label ?flapLabel }
  OPTIONAL { ?bloodSupplyCls rdfs:label ?bloodSupplyLabel }
}
"""
run(_cq8.replace("PAIRS", vasc_pairs),
    key="CQ8", title="CQ8 ABox — flap individuals by vascular pattern")

CQ8 ABox — flap individuals by vascular pattern
  138 individuals returned, 808 rows (first 10 shown)


,flapLabel,bloodSupplyLabel
0,Left Vy flaps of the fingertip flap with skin ...,Random pattern flaps
1,Right Vy flaps of the fingertip flap with skin...,Random pattern flaps
2,Left Vy flaps of the fingertip flap with skin ...,Random pattern flaps
3,Left Vy flaps of the fingertip flap with skin ...,Random pattern flaps
4,Right Vy flaps of the fingertip flap with skin...,Random pattern flaps
5,Left Vy flaps of the fingertip flap with skin ...,Random pattern flaps
6,Right Vy flaps of the fingertip flap with skin...,Random pattern flaps
7,Left Vy flaps of the fingertip flap with skin ...,Random pattern flaps
8,Right Vy flaps of the fingertip flap with skin...,Random pattern flaps
9,Right Vy flaps of the fingertip flap with skin...,Random pattern flaps


### CQ9 — Chimeric Flap
*Is it a chimeric flap?*

In [13]:
_cq9 = """
SELECT ?flapLabel ?chimericTypeLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?chimericTypeLabel }
}
"""
run(_cq9.replace("CLS", chimeric_cls),
    key="CQ9", title="CQ9 ABox — flap individuals by chimeric type")

CQ9 ABox — flap individuals by chimeric type
  16 individuals returned, 32 rows (first 10 shown)


,flapLabel,chimericTypeLabel
0,Chimeric Supraclavicular artery flap with subc...,Flaps classified by chimeric type
1,Chimeric Supraclavicular artery flap with skin...,Flaps classified by chimeric type
2,Chimeric Temporoparietal fascia flap with temp...,Flaps classified by chimeric type
3,Chimeric Temporalis flap of patient 2003,Flaps classified by chimeric type
4,Chimeric Auricularis posterior flap of patient...,Flaps classified by chimeric type
5,Chimeric Paramedian forehead flap of patient 2002,Flaps classified by chimeric type
6,Chimeric Nasolabial flap of patient 2001,Flaps classified by chimeric type
7,Chimeric Omohyoid flap with fascia of neck of ...,Flaps classified by chimeric type
8,Chimeric Omohyoid flap with ansa cervicalis of...,Flaps classified by chimeric type
9,Chimeric Omohyoid flap with sternocleidomastoi...,Flaps classified by chimeric type


### CQ10 — Arterial Flow Direction
*What is the arterial flow direction (e.g. reversed flow or flow-through flap)?*

In [14]:
_cq10 = """
SELECT DISTINCT ?flap ?flapLabel ?type ?typeLabel ?rootLabel
WHERE {
  VALUES ?root { CLS }

  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?type .

  ?type rdfs:subClassOf* ?root .

  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?type rdfs:label ?typeLabel }
  OPTIONAL { ?root rdfs:label ?rootLabel }
}
"""

run(
    _cq10.replace("CLS", _cls_vals(OFL.OFLID10093)),
    key="CQ10", title="CQ10 ABox — individuals under OFLID10093"
)

CQ10 ABox — individuals under OFLID10093
  302 individuals returned, 1,812 rows (first 10 shown)


,flap,flapLabel,type,typeLabel,rootLabel
0,https://purl.bioontology.org/ontology/OFL/OFLI...,Left Frontalis muscle flap of patient 85,c494ab4b41ad78b3d719645fd9203915,—,Flaps classified by arterial flow direction
1,https://purl.bioontology.org/ontology/OFL/OFLI...,Right Inferior gluteal artery perforator flap ...,c494ab4b41ad78b3d719645fd9203915,—,Flaps classified by arterial flow direction
2,https://purl.bioontology.org/ontology/OFL/OFLI...,Right Nasolabial flap of patient 6,c494ab4b41ad78b3d719645fd9203915,—,Flaps classified by arterial flow direction
3,https://purl.bioontology.org/ontology/OFL/OFLI...,Right Medial sural artery perforator flap of p...,c494ab4b41ad78b3d719645fd9203915,—,Flaps classified by arterial flow direction
4,https://purl.bioontology.org/ontology/OFL/OFLI...,Right Paramedian forehead flap of patient 20,c494ab4b41ad78b3d719645fd9203915,—,Flaps classified by arterial flow direction
5,https://purl.bioontology.org/ontology/OFL/OFLI...,Right Gluteal thigh flap (cutaneous island) of...,c494ab4b41ad78b3d719645fd9203915,—,Flaps classified by arterial flow direction
6,https://purl.bioontology.org/ontology/OFL/OFLI...,Left Inferior gluteal artery perforator flap o...,c494ab4b41ad78b3d719645fd9203915,—,Flaps classified by arterial flow direction
7,https://purl.bioontology.org/ontology/OFL/OFLI...,Left First dorsal metacarpal artery flap of pa...,c494ab4b41ad78b3d719645fd9203915,—,Flaps classified by arterial flow direction
8,https://purl.bioontology.org/ontology/OFL/OFLI...,Right Superior gluteal artery perforator flap ...,c494ab4b41ad78b3d719645fd9203915,—,Flaps classified by arterial flow direction
9,https://purl.bioontology.org/ontology/OFL/OFLI...,Left Medial sural artery perforator flap of pa...,c494ab4b41ad78b3d719645fd9203915,—,Flaps classified by arterial flow direction


### CQ11 — Movement Type
*Is it a rotational, advancement or transpositional flap?*

In [15]:
_cq11 = """
SELECT ?flapLabel ?movementLabel
WHERE {
  VALUES (?movement ?t) { PAIRS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap     rdfs:label ?flapLabel }
  OPTIONAL { ?movement rdfs:label ?movementLabel }
}
"""
run(_cq11.replace("PAIRS", movement_pairs),
    key="CQ11", title="CQ11 ABox — flap individuals by movement type")

CQ11 ABox — flap individuals by movement type
  3,452 individuals returned, 3,815 rows (first 10 shown)


,flapLabel,movementLabel
0,Left Iliac crest flap with transversus abdomin...,Rotation flaps
1,Right Iliac crest flap with transversus abdomi...,Rotation flaps
2,Right Gastrocnemius flap with subcutaneous adi...,Rotation flaps
3,Left Latissimus dorsi flap with skin of back o...,Rotation flaps
4,Left Dorsal metacarpal artery flap with subcut...,Rotation flaps
5,Right Superficial sural artery flap of patient...,Rotation flaps
6,Left Pectoralis major flap with clavicle of pa...,Rotation flaps
7,Right Paramedian forehead flap of patient 20,Rotation flaps
8,Right Groin flap with ilium of patient 379,Rotation flaps
9,Right Medial femoral condyle flap with medial ...,Rotation flaps


### CQ12 — Island Flap
*Is it an island flap?*

In [16]:
#island flap class: OFLID120032
#Surgical flap class: OFLID10002

_cq12 = """
SELECT DISTINCT ?flap ?flapLabel ?islandLabel
WHERE {
  VALUES ?t { CLS }

  ?flap rdf:type ?t .

  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t rdfs:label ?islandLabel }
}
"""

run(
    _cq12.replace("CLS", _cls_vals(OFL.OFLID120032)),
    key="CQ12", title="CQ12 ABox — direct island flap individuals"
)

CQ12 ABox — direct island flap individuals
  22 individuals returned, 44 rows (first 10 shown)


,flap,flapLabel,islandLabel
0,https://purl.bioontology.org/ontology/OFL/OFLI...,Right Gluteal thigh flap (cutaneous island) of...,Cutaneous island flap
1,https://purl.bioontology.org/ontology/OFL/OFLI...,Left Auricularis posterior flap (cutaneous isl...,Cutaneous island flap
2,https://purl.bioontology.org/ontology/OFL/OFLI...,Right Auricularis posterior flap (cutaneous is...,Cutaneous island flap
3,https://purl.bioontology.org/ontology/OFL/OFLI...,Left Medial plantar flap (cutaneous island) of...,Cutaneous island flap
4,https://purl.bioontology.org/ontology/OFL/OFLI...,Left Gluteal thigh flap (cutaneous island) of ...,Cutaneous island flap
5,https://purl.bioontology.org/ontology/OFL/OFLI...,Right Paramedian forehead flap (cutaneous isla...,Cutaneous island flap
6,https://purl.bioontology.org/ontology/OFL/OFLI...,Left Nasolabial flap (cutaneous island) of pat...,Cutaneous island flap
7,https://purl.bioontology.org/ontology/OFL/OFLI...,Right First dorsal metacarpal artery flap (cut...,Cutaneous island flap
8,https://purl.bioontology.org/ontology/OFL/OFLI...,Left Great toe pulp flap (cutaneous island) of...,Cutaneous island flap
9,https://purl.bioontology.org/ontology/OFL/OFLI...,Right Frontalis muscle flap (cutaneous island)...,Cutaneous island flap


### CQ13 — Insertion Site Preparation
*How was the insertion site prepared?*

In [17]:
_cq13 = """
SELECT ?flapLabel ?prepTypeLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?prepTypeLabel }
}
"""
run(_cq13.replace("CLS", insertion_prep_cls),
    key="CQ13", title="CQ13 ABox — flap individuals by insertion site preparation type")

CQ13 ABox — flap individuals by insertion site preparation type
  3,932 individuals returned, 3,932 rows (first 10 shown)


,flapLabel,prepTypeLabel
0,Right Vertical rectus abdominis flap of patien...,Flaps classified by insertion site preparation
1,Left Pectoralis major flap with sternum of pat...,Flaps classified by insertion site preparation
2,Left Iliac crest flap with transversus abdomin...,Flaps classified by insertion site preparation
3,Right Groin flap with skin of mastoid part of ...,Flaps classified by insertion site preparation
4,Left Pectoralis major flap with pectoral fasci...,Flaps classified by insertion site preparation
5,Right Parascapular flap with skin of back of p...,Flaps classified by insertion site preparation
6,Right Iliac crest flap with transversus abdomi...,Flaps classified by insertion site preparation
7,Right Parascapular flap with teres minor of pa...,Flaps classified by insertion site preparation
8,Right Second toe transfer of patient 1147,Flaps classified by insertion site preparation
9,Left Serratus anterior muscle flap with subcut...,Flaps classified by insertion site preparation


### CQ14 — Pre-Harvest Modification
*Was the flap modified before harvesting?*

In [18]:
_cq14 = """
SELECT ?flapLabel ?modLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?modLabel }
}
"""
run(_cq14.replace("CLS", preharvest_cls),
    key="CQ14", title="CQ14 ABox — pre-harvest modified flap individuals")

CQ14 ABox — pre-harvest modified flap individuals
  3,948 individuals returned, 7,896 rows (first 10 shown)


,flapLabel,modLabel
0,Right Vertical rectus abdominis flap of patien...,Flaps classified by pre harvest modification
1,Left Pectoralis major flap with sternum of pat...,Flaps classified by pre harvest modification
2,Left Iliac crest flap with transversus abdomin...,Flaps classified by pre harvest modification
3,Right Groin flap with skin of mastoid part of ...,Flaps classified by pre harvest modification
4,Left Pectoralis major flap with pectoral fasci...,Flaps classified by pre harvest modification
5,Right Parascapular flap with skin of back of p...,Flaps classified by pre harvest modification
6,Right Iliac crest flap with transversus abdomi...,Flaps classified by pre harvest modification
7,Right Parascapular flap with teres minor of pa...,Flaps classified by pre harvest modification
8,Right Second toe transfer of patient 1147,Flaps classified by pre harvest modification
9,Left Serratus anterior muscle flap with subcut...,Flaps classified by pre harvest modification


### CQ15 — Split-Thickness Skin Grafting
*Did the flap require split-thickness skin grafting?*

In [19]:
_cq15 = """
SELECT ?flapLabel ?graftTypeLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?graftTypeLabel }
}
"""
run(_cq15.replace("CLS", skingraft_cls),
    key="CQ15", title="CQ15 ABox — flap individuals requiring split-thickness skin grafting")

CQ15 ABox — flap individuals requiring split-thickness skin grafting
  1,964 individuals returned, 1,964 rows (first 10 shown)


,flapLabel,graftTypeLabel
0,Right Vertical rectus abdominis flap of patien...,Flaps with skin graft
1,Left Pectoralis major flap with sternum of pat...,Flaps with skin graft
2,Left Iliac crest flap with transversus abdomin...,Flaps with skin graft
3,Right Parascapular flap with skin of back of p...,Flaps with skin graft
4,Left Radial forearm flap with antebrachial fas...,Flaps with skin graft
5,Left Latissimus dorsi flap with skin of back o...,Flaps with skin graft
6,Right Soleus flap with skin of lateral part of...,Flaps with skin graft
7,Left Pectoralis major flap with clavicle of pa...,Flaps with skin graft
8,Right Gluteal artery perforator flap with glut...,Flaps with skin graft
9,Right Nasolabial flap of patient 6,Flaps with skin graft


### CQ16 — Vessel Anastomosis
*How was vessel anastomosis performed?*

In [20]:
_cq16 = """
SELECT ?anastLabel ?anastomosisTypeLabel
WHERE {
  VALUES ?t { CLS }
  ?anast rdf:type owl:NamedIndividual ;
         rdf:type ?t .
  OPTIONAL { ?anast rdfs:label ?anastLabel }
  OPTIONAL { ?t     rdfs:label ?anastomosisTypeLabel }
}
"""
run(_cq16.replace("CLS", anast_cls),
    key="CQ16", count_var="?anast", title="CQ16 ABox — vessel anastomosis individuals by type")

CQ16 ABox — vessel anastomosis individuals by type
  960 individuals returned, 5,760 rows (first 10 shown)


,anastLabel,anastomosisTypeLabel
0,Arterial anastomosis of Right Gluteal artery p...,Surgically produced vessel anastomosis
1,Arterial anastomosis of Left Lateral thoracic ...,Surgically produced vessel anastomosis
2,Arterial anastomosis of Left Thoracodorsal art...,Surgically produced vessel anastomosis
3,Venous anastomosis of Right Superior gluteal a...,Surgically produced vessel anastomosis
4,Arterial anastomosis of Left Serratus anterior...,Surgically produced vessel anastomosis
5,Arterial anastomosis of Right Thoracodorsal ar...,Surgically produced vessel anastomosis
6,Arterial anastomosis of Right Rectus abdominis...,Surgically produced vessel anastomosis
7,Arterial anastomosis of Left Serratus anterior...,Surgically produced vessel anastomosis
8,Arterial anastomosis of Right Medial sural art...,Surgically produced vessel anastomosis
9,Arterial anastomosis of Left Serratus anterior...,Surgically produced vessel anastomosis


### CQ17 — Flap Survival
*Did the flap fully survive?*

In [21]:
run("""
SELECT ?flapLabel ?outcomeLabel
WHERE {
  VALUES ?survCls { ofl:OFLID10184 ofl:OFLID10183 ofl:OFLID10179
                   ofl:OFLID10178 ofl:OFLID10180 ofl:OFLID10181 }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?survCls .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?survCls rdfs:label ?outcomeLabel }
}
""", key="CQ17", title="CQ17 ABox — flap individuals by survival outcome")

CQ17 ABox — flap individuals by survival outcome
  3,948 individuals returned, 4,742 rows (first 10 shown)


,flapLabel,outcomeLabel
0,Left Iliac crest flap with transversus abdomin...,Flaps without tissue loss
1,Right Iliac crest flap with transversus abdomi...,Flaps without tissue loss
2,Right Parascapular flap with teres minor of pa...,Flaps without tissue loss
3,Left Radial forearm flap with antebrachial fas...,Flaps without tissue loss
4,Right Gastrocnemius flap with subcutaneous adi...,Flaps without tissue loss
5,Left Radial forearm flap with skin of anterior...,Flaps without tissue loss
6,Right Inferior gluteal artery perforator flap ...,Flaps without tissue loss
7,Right Nasolabial flap of patient 6,Flaps without tissue loss
8,Left Jejunum flap with mesenteric lymph node o...,Flaps without tissue loss
9,Right Trapezius flap with subcutaneous adipose...,Flaps without tissue loss


---
## Task-Based Evaluation

Five realistic clinical and research tasks demonstrating the ontology supports end-to-end use cases beyond abstract CQ answering.

### Task 1 — Flap selection by donor region
*Pedicled flap for trunk reconstruction — which flap types originate from the back of the trunk?*

In [22]:
_task1 = """
SELECT ?flapLabel ?survivalLabel
WHERE {
  VALUES ?procType { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        ofl:OFLID12003 ?origin .
  ?origin rdfs:label ?originLabel .
  FILTER(CONTAINS(LCASE(?originLabel), "back of trunk"))
  ?flap obo:RO_0000056 ?process .
  ?process rdf:type ?procType .
  VALUES ?survCls { ofl:OFLID10184 ofl:OFLID10183 ofl:OFLID10179
                   ofl:OFLID10178 ofl:OFLID10180 ofl:OFLID10181 }
  ?flap rdf:type ?survCls .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?survCls rdfs:label ?survivalLabel }
}
"""
run(_task1.replace("CLS", pedicled_cls),
    key="Task 1", title="Task 1 — Pedicled back-of-trunk flap individuals with survival outcome")

Task 1 — Pedicled back-of-trunk flap individuals with survival outcome
  20 individuals returned, 62 rows (first 10 shown)


,flapLabel,survivalLabel
0,Left Latissimus dorsi flap with subcutaneous a...,Flaps with loss at the apex
1,Left Latissimus dorsi flap with subcutaneous a...,Flaps with partial loss
2,Right Latissimus dorsi flap with subcutaneous ...,Flaps without tissue loss
3,Left Latissimus dorsi flap with subcutaneous a...,Flaps without tissue loss
4,Right Latissimus dorsi flap with subcutaneous ...,Flaps with venous obstruction
5,Left Latissimus dorsi flap with subcutaneous a...,Flaps with arterial obstruction
6,Left Latissimus dorsi flap with subcutaneous a...,Flaps with venous obstruction
7,Left Latissimus dorsi flap with subcutaneous a...,Flaps with loss at the apex
8,Left Latissimus dorsi flap with subcutaneous a...,Flaps with partial loss
9,Left Latissimus dorsi flap with subcutaneous a...,Flaps without tissue loss


### Task 2 — Free flap survival audit
*Retrieve free flap individuals with their survival outcome.*

In [23]:
run("""
SELECT ?flapLabel ?survivalLabel
WHERE {
  ?flap rdf:type owl:NamedIndividual ;
        obo:RO_0000056 ?process .
  ?process rdf:type ofl:OFLID1000137 .
  VALUES ?survCls { ofl:OFLID10184 ofl:OFLID10183 ofl:OFLID10179
                   ofl:OFLID10178 ofl:OFLID10180 ofl:OFLID10181 }
  ?flap rdf:type ?survCls .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?survCls rdfs:label ?survivalLabel }
}
""", key="Task 2", title="Task 2 — Free flap individuals with survival outcome")

Task 2 — Free flap individuals with survival outcome
  480 individuals returned, 576 rows (first 10 shown)


,flapLabel,survivalLabel
0,Right Inferior gluteal artery perforator flap ...,Flaps without tissue loss
1,Left Posterior tibial artery perforator flap w...,Flaps without tissue loss
2,Left Inferior gluteal artery perforator flap o...,Flaps without tissue loss
3,Left Serratus anterior perforator flap with su...,Flaps without tissue loss
4,Right Thoracodorsal artery perforator flap wit...,Flaps without tissue loss
5,Left Deep inferior epigastric perforator flap ...,Flaps without tissue loss
6,Right Posterior tibial artery perforator flap ...,Flaps without tissue loss
7,Right Thoracodorsal artery perforator flap wit...,Flaps without tissue loss
8,Left Thoracodorsal artery perforator flap with...,Flaps without tissue loss
9,Right Gluteal artery perforator flap with skin...,Flaps without tissue loss


### Task 3 — Mathes-Nahai Type I flap lookup
*Which muscle flap individuals have a single dominant vascular pedicle (Mathes-Nahai Type I)?*

In [24]:
_task3 = """
SELECT ?flapLabel ?mnTypeLabel
WHERE {
  VALUES (?mnType ?flapType) { PAIRS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?flapType .
  OPTIONAL { ?mnType rdfs:label ?mnTypeLabel }
  OPTIONAL { ?flap   rdfs:label ?flapLabel }
}
"""
run(_task3.replace("PAIRS", mn_task3_pairs),
    key="Task 3", title="Task 3 — ABox flap individuals by Mathes-Nahai vascular pattern type")

Task 3 — ABox flap individuals by Mathes-Nahai vascular pattern type
  1,246 individuals returned, 4,982 rows (first 10 shown)


,flapLabel,mnTypeLabel
0,Right Oblique rectus abdominis flap of patient...,Oblique rectus abdominis flap
1,Right Oblique rectus abdominis flap of patient...,Oblique rectus abdominis flap
2,Left Oblique rectus abdominis flap of patient ...,Oblique rectus abdominis flap
3,Left Oblique rectus abdominis flap of patient ...,Oblique rectus abdominis flap
4,Left Oblique rectus abdominis flap of patient ...,Oblique rectus abdominis flap
5,Right Oblique rectus abdominis flap of patient...,Oblique rectus abdominis flap
6,Left Rectus abdominis flap with fascia of rect...,Muscle flaps classified by Mathes and Nahai
7,Right Rectus abdominis flap with fascia of rec...,Muscle flaps classified by Mathes and Nahai
8,Left Rectus abdominis flap with fascia of rect...,Muscle flaps classified by Mathes and Nahai
9,Left Rectus abdominis flap with fascia of rect...,Muscle flaps classified by Mathes and Nahai


### Task 4 — Prefabrication outcome tracking
*Pre-harvest modified flap individuals cross-tabulated with survival outcome.*

In [25]:
run("""
SELECT ?flapLabel ?survivalLabel
WHERE {
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ofl:OFLID10131 .
  VALUES ?survCls { ofl:OFLID10184 ofl:OFLID10183 ofl:OFLID10179
                   ofl:OFLID10178 ofl:OFLID10180 ofl:OFLID10181 }
  ?flap rdf:type ?survCls .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?survCls rdfs:label ?survivalLabel }
}
""", key="Task 4", title="Task 4 — Survival outcomes for pre-harvest modified flaps")

Task 4 — Survival outcomes for pre-harvest modified flaps
  1,910 individuals returned, 2,292 rows (first 10 shown)


,flapLabel,survivalLabel
0,Right Parascapular flap with teres minor of pa...,Flaps without tissue loss
1,Right Gastrocnemius flap with subcutaneous adi...,Flaps without tissue loss
2,Left Radial forearm flap with skin of anterior...,Flaps without tissue loss
3,Left Jejunum flap with mesenteric lymph node o...,Flaps without tissue loss
4,Left Posterior tibial artery perforator flap w...,Flaps without tissue loss
5,Right Iliac crest flap with internal oblique o...,Flaps without tissue loss
6,Right Scapular flap with subcutaneous adipose ...,Flaps without tissue loss
7,Right Gluteus flap with gluteus maximus of pat...,Flaps without tissue loss
8,Left Inferior gluteal artery perforator flap o...,Flaps without tissue loss
9,Left Tensor fasciae latae flap with iliac cres...,Flaps without tissue loss


### Task 5 — Operative planning: anatomical component inventory
*Component individuals of a generated Fibula flap instance with their FMA types.*

In [26]:
run("""
SELECT ?componentLabel ?typeLabel
WHERE {
  ?flap rdfs:label ?fl . FILTER(CONTAINS(LCASE(?fl), "fibula flap"))
  ?flap ofl:OFLID13296 ?component .
  OPTIONAL { ?component rdfs:label ?componentLabel }
  OPTIONAL {
    ?component rdf:type ?t .
    FILTER(STRSTARTS(STR(?t), "http://purl.obolibrary.org/obo/FMA_"))
    OPTIONAL { ?t rdfs:label ?typeLabel }
  }
}
""", key="Task 5", count_var="?component", title="Task 5 — Component individuals of a generated Fibula flap ABox instance")

Task 5 — Component individuals of a generated Fibula flap ABox instance
  1,000 individuals returned, 1,000 rows (first 10 shown)


,componentLabel,typeLabel
0,Flap tissue of Left Fibula flap with small sap...,Anatomical structure
1,Left structure 44333 of patient 1185,Small saphenous vein
2,Flap harvest process of Right Fibula flap with...,—
3,Pedicled flap transfer process of Right Fibula...,—
4,Preinsertion treatment of recipient site for R...,—
5,Flap tissue of Right Fibula flap with posterio...,Anatomical structure
6,Right structure 58758 of patient 1240,Posterior intermuscular septum of leg
7,Flap tissue of Left Fibula flap with fascia of...,Anatomical structure
8,Left structure 47188 of patient 1215,Fascia of flexor hallucis longus
9,Flap tissue of Left Fibula flap with tibialis ...,Anatomical structure


---
## Summary

Individuals returned and result rows per query (Supplementary Table S2).

In [27]:
summary = pd.DataFrame.from_dict(RESULTS, orient='index')
summary.index.name = 'Query'
summary

,Individuals returned,Rows
Query,,
CQ1,3948,231692
CQ2,3948,191356
CQ3,302,27614
CQ4,1240,2888
CQ5,60,180
CQ6,6,6
CQ7,140,560
CQ8,138,808
CQ9,16,32
